In [6]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def calculate_avm_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Compute headline (MdAPE) and complementary metrics (R2, MAPE, MAE, RMSE)

    for AVM evaluation on dollar-scale targets.
    """
    abs_errors = np.abs(y_true - y_pred)
    percentage_errors = abs_errors / y_true

    mdape = float(np.median(percentage_errors) * 100)
    mape = float(np.mean(percentage_errors) * 100)
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))

    return {
        "MdAPE (%)": round(mdape, 2),
        "MAPE (%)": round(mape, 2),
        "MAE ($)": round(mae, 2),
        "RMSE ($)": round(rmse, 2),
        "R2": round(r2, 4),
    }


def run_baseline_model(data_dir: str | Path, output_dir: str | Path) -> pd.DataFrame:
    """Trains a baseline Linear Regression model, evaluates on validation and test sets,

    and exports benchmark results.
    """
    data_dir = Path(data_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Load processed features and target vectors
    X_train = pd.read_parquet(data_dir / "X_train_processed.parquet")
    X_val = pd.read_parquet(data_dir / "X_val_processed.parquet")
    X_test = pd.read_parquet(data_dir / "X_test_processed.parquet")

    y_train = pd.read_parquet(data_dir / "y_train.parquet")["ClosePrice"].to_numpy()
    y_val = pd.read_parquet(data_dir / "y_val.parquet")["ClosePrice"].to_numpy()
    y_test = pd.read_parquet(data_dir / "y_test.parquet")["ClosePrice"].to_numpy()

    # 2. Log-transform target to stabilize variance and prevent negative predictions
    y_train_log = np.log1p(y_train)

    # 3. Fit Linear Regression baseline model on training data only
    model = LinearRegression()
    model.fit(X_train, y_train_log)

    # 4. Predict and invert log transformation back to dollar scale
    y_train_pred = np.expm1(model.predict(X_train))
    y_val_pred = np.expm1(model.predict(X_val))
    y_test_pred = np.expm1(model.predict(X_test))

    # 5. Compute performance metrics on dollar scale
    metrics_train = calculate_avm_metrics(y_train, y_train_pred)
    metrics_val = calculate_avm_metrics(y_val, y_val_pred)
    metrics_test = calculate_avm_metrics(y_test, y_test_pred)

    results_df = pd.DataFrame(
        [metrics_train, metrics_val, metrics_test],
        index=["Train", "Validation", "Test"],
    )

    # 6. Save baseline metrics to disk
    results_path = output_dir / "baseline_linear_regression_metrics.json"
    with open(results_path, "w") as f:
        json.dump(
            {
                "model_name": "LinearRegression_Baseline",
                "metrics": results_df.to_dict(orient="index"),
            },
            f,
            indent=4,
        )

    print("--- Baseline Linear Regression Evaluation Results ---")
    print(results_df.to_string())

    return results_df


if __name__ == "__main__":
    DATA_DIR = Path("model_artifacts")
    OUTPUT_DIR = Path("models/baseline")
    run_baseline_model(DATA_DIR, OUTPUT_DIR)

--- Baseline Linear Regression Evaluation Results ---
            MdAPE (%)  MAPE (%)    MAE ($)    RMSE ($)      R2
Train           11.94     16.31  212569.38   630273.28  0.5243
Validation      12.23     16.50  227634.47  1002464.16 -0.1681
Test            12.16     16.31  223498.65   517147.61  0.7071
